# Hotel Booking Demand Analysis
## Machine Learning
* **Goal:** Predict Booking Cancellations with two datasats (Dataset Comparison)

In [18]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

In [19]:
# Import datasets with try/except

file_with_dup = 'hotel_booking_with_dup_ml.csv'
file_no_dup = 'hotel_booking_no_dup_ml.csv'

# Load dataset with duplicates
try:
    df_with_dup = pd.read_csv(file_with_dup)
    print(f'{file_with_dup} loaded successfully')
except FileNotFoundError:
    print(f'File not found - check {df_with_dup}')

# Load dataset without duplicates
try:
    df_no_dup = pd.read_csv(file_no_dup)
    print(f'{file_no_dup} loaded successfully')
except FileNotFoundError:
    print(f'File not found - check {df_no_dup}')

hotel_booking_with_dup_ml.csv loaded successfully
hotel_booking_no_dup_ml.csv loaded successfully


In [20]:
# Implement function to check the head of both datasets

def inspection_head(df, name):
    """
    Inspection function to analyze dataset shape and header
    """
    print(f'Dataset: {name}')
    print(f'Rows: {df.shape[0]} -- Columns: {df.shape[1]}')
    display(df.head())
    return df

In [21]:
df_with_dup = inspection_head(df_with_dup, 'with Duplicates')
df_no_dup = inspection_head(df_no_dup, 'no Duplicates')

Dataset: with Duplicates
Rows: 119388 -- Columns: 54


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,737,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,7,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
3,0,13,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,14,2015,27,1,0,2,2,0.0,0,...,0,1,0,1,0,0,0,0,1,0


Dataset: no Duplicates
Rows: 87394 -- Columns: 54


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,342,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
1,0,737,2015,27,1,0,0,2,0.0,0,...,0,0,0,1,0,0,0,0,1,0
2,0,7,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
3,0,13,2015,27,1,0,1,1,0.0,0,...,0,0,0,1,0,0,0,0,1,0
4,0,14,2015,27,1,0,2,2,0.0,0,...,0,1,0,1,0,0,0,0,1,0


In [22]:
# Splitting both datasets into X the features and y the target
def prepare_model_data(df, target_col='is_canceled'):
    """
    Splitting the dataset for train_test_model into X the features and y the target
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test

In [40]:
# Call function for splitting the dataset into train_test
# Call function for dataset with duplicates
X_train_wd, X_test_wd, X_train_scaled_wd, X_test_scaled_wd, y_train_wd, y_test_wd = prepare_model_data(df_with_dup)

# Call function for dataset without duplicates
X_train_nd, X_test_nd, X_train_scaled_nd, X_test_scaled_nd, y_train_nd, y_test_nd = prepare_model_data(df_no_dup)

print('Preparing and Scaling complete')
# Check for the shape 
print('=' * 60)
print('Dataset Shape View')
print('=' * 60)
print(f'---Dataset without Duplicates---')
print(f'Trainset:\nRows:{X_train_nd.shape[0]} - Cols:{X_train_nd.shape[1]}\nTestset:\nRows:{X_test_nd.shape[0]}')
print(f'---Dataset with Duplicates---')
print(f'Trainset:\nRows:{X_train_wd.shape[0]} - Cols:{X_train_wd.shape[1]}\nTestset:\nRows:{X_test_wd.shape[0]}')


Preparing and Scaling complete
Dataset Shape View
---Dataset without Duplicates---
Trainset:
Rows:69915 - Cols:53
Testset:
Rows:17479
---Dataset with Duplicates---
Trainset:
Rows:95510 - Cols:53
Testset:
Rows:23878


In [47]:
# Create LogisticRegression Model for first prediction with standard hyperparamter
# Create first model for Dataset with duplicates
logreg_wd = LogisticRegression(max_iter= 1000, random_state=42)

# Train the model
logreg_wd.fit(X_train_scaled_wd, y_train_wd)

# Make prediction
logreg_wd_pred = logreg_wd.predict(X_test_scaled_wd)

print("--- Results: Logistic Regression (WITH Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_wd, logreg_wd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_wd, logreg_wd_pred))

--- Results: Logistic Regression (WITH Duplicates) ---
Accuracy: 81.13

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86     15033
           1       0.80      0.65      0.72      8845

    accuracy                           0.81     23878
   macro avg       0.81      0.78      0.79     23878
weighted avg       0.81      0.81      0.81     23878



In [48]:
# Create LogisticRegression Model for first prediction with standard hyperparamter
# Create first model for Dataset without duplicates
logreg_nd = LogisticRegression(max_iter= 1000, random_state=42)

# Train the model
logreg_nd.fit(X_train_scaled_nd, y_train_nd)

# Make prediction
logreg_nd_pred = logreg_nd.predict(X_test_scaled_nd)

print("--- Results: Logistic Regression (WITHOUT Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_nd, logreg_nd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_nd, logreg_nd_pred))

--- Results: Logistic Regression (WITHOUT Duplicates) ---
Accuracy: 78.81

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.91      0.86     12674
           1       0.66      0.47      0.55      4805

    accuracy                           0.79     17479
   macro avg       0.74      0.69      0.70     17479
weighted avg       0.78      0.79      0.78     17479



### LogisticRegression Model Evaluation: The Impact of Data Duplication

When comparing the Logistic Regression performance between the two datasets, the overall **Accuracy** only shows a minor difference (**81.13%** with duplicates vs. **78.81%** without duplicates) despite the removal of over 30,000 rows. However, a deeper dive into the classification report reveals a massive impact on predicting actual cancellations (**Class 1**):

* **Recall (Class 1):** Dropped significantly from **65% to 47%**.
* **Precision (Class 1):** Decreased from **80% to 66%**.
* **F1-Score (Class 1):** Fell sharply from **72% to 55%**.
* **Class 0 Metrics (Not Canceled):** Remained incredibly stable across both datasets (Recall at 91%, F1 at 86%).

**Key Takeaway:**
The model trained *with* duplicates was heavily overestimating its ability to predict cancellations. It was likely overfitting by memorizing repeated identical rows (e.g., bulk group bookings). The dataset *without* duplicates exposes the model's true, unbiased ability to generalize to new, unseen data. Moving forward, the deduplicated dataset will serve as the realistic benchmark for our more complex models.

In [51]:
# Create RandomForest Model for first prediction with standard hyperparamter
# Create first model for Dataset with duplicates
rf_model_wd = RandomForestClassifier(random_state=42)

# Train the model
rf_model_wd.fit(X_train_wd, y_train_wd)

# Make prediction
rf_model_wd_pred = rf_model_wd.predict(X_test_wd)

print("--- Results: Random Forest (WITH Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_wd, rf_model_wd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_wd, rf_model_wd_pred))

--- Results: Random Forest (WITH Duplicates) ---
Accuracy: 88.84

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     15033
           1       0.88      0.81      0.84      8845

    accuracy                           0.89     23878
   macro avg       0.89      0.87      0.88     23878
weighted avg       0.89      0.89      0.89     23878



In [ ]:
# Create RandomForest Model for first prediction with standard hyperparamter
# Create first model for Dataset without duplicates
rf_model_nd = RandomForestClassifier(random_state=42)

# Train the model
rf_model_nd.fit(X_train_nd, y_train_nd)

# Make prediction
rf_model_nd_pred = rf_model_nd.predict(X_test_nd)

print("--- Results: Random Forest (WITHOUT Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_nd, rf_model_nd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_nd, rf_model_nd_pred))

--- Results: Random Forest (WITHOUT Duplicates) ---
Accuracy: 84.15

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.92      0.89     12674
           1       0.75      0.64      0.69      4805

    accuracy                           0.84     17479
   macro avg       0.81      0.78      0.79     17479
weighted avg       0.84      0.84      0.84     17479



### Random Forest Model Evaluation: 

Transitioning from the linear baseline to a non-linear **Random Forest Classifier** yields a massive performance boost across both datasets. Because tree-based ensembles can naturally capture complex "if-then" feature interactions, the model handles the prediction task much more effectively. 

However, the contrast between the two datasets remains highly visible, further validating our deduplication choice:

* **Overall Accuracy:** Decreased from **88.84%** (with duplicates) to **84.15%** (without duplicates).
* **Recall (Class 1 - Cancellations):** Dropped from an inflated **81% down to 64%** once identical repeated rows were removed.
* **Precision (Class 1):** Shifted from **88% to 75%**.
* **F1-Score (Class 1):** Fell from **84% to 69%**.

#### Comparison to Logistic Regression (On Deduplicated Data):
Even though removing duplicates forces the model to work with a harder, unbiased dataset, the Random Forest completely destroys the Logistic Regression baseline on the realistic data:
* **Recall (Class 1):** Surged from **47% to 64%** (+17% absolute improvement).
* **Precision (Class 1):** Increased from **66% to 75%** (+9% absolute improvement).
* **F1-Score (Class 1):** Jumped from **55% to 69%** (+14% absolute improvement).

**Key Takeaway:**
While the dataset *with* duplicates continues to show artificially inflated metrics due to row memorization, the **84.15% accuracy** and **64% recall** on the deduplicated dataset represent a highly capable, robust, and realistic model. The Random Forest successfully captures the non-linear dynamics of hotel cancellations without relying on artificial data inflation.

In [55]:
# Create XGBoost Model for first prediction with standard hyperparamter
# Create first model for Dataset with duplicates
xgboost_wd = XGBClassifier(random_state = 42, n_estimators=200, learning_rate=0.1)

# Train the model
xgboost_wd.fit(X_train_wd, y_train_wd)

# Make prediction
xgboost_wd_pred = xgboost_wd.predict(X_test_wd)

print("--- Results: XGBoost (WITH Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_wd, xgboost_wd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_wd, xgboost_wd_pred))

--- Results: XGBoost (WITH Duplicates) ---
Accuracy: 86.88

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     15033
           1       0.85      0.79      0.82      8845

    accuracy                           0.87     23878
   macro avg       0.86      0.85      0.86     23878
weighted avg       0.87      0.87      0.87     23878



In [56]:
# Create XGBoost Model for first prediction with standard hyperparamter
# Create first model for Dataset without duplicates
xgboost_nd = XGBClassifier(random_state = 42, n_estimators=200, learning_rate=0.1)

# Train the model
xgboost_nd.fit(X_train_nd, y_train_nd)

# Make prediction
xgboost_nd_pred = xgboost_nd.predict(X_test_nd)

print("--- Results: XGBoost (WITHOUT Duplicates) ---")
print(f"Accuracy: {accuracy_score(y_test_nd, xgboost_nd_pred)*100:.2f}")
print("\nClassification Report:")
print(classification_report(y_test_nd, xgboost_nd_pred))

--- Results: XGBoost (WITHOUT Duplicates) ---
Accuracy: 84.02

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.91      0.89     12674
           1       0.74      0.65      0.69      4805

    accuracy                           0.84     17479
   macro avg       0.81      0.78      0.79     17479
weighted avg       0.84      0.84      0.84     17479



### Extreme Gradient Boosting (XGBoost) Evaluation

As final baseline comparison, we trained an **XGBoost Classifier** (`n_estimators=200`, `learning_rate=0.1`). Gradient boosting algorithms build trees sequentially, learning from the errors of previous trees, which often provides a slight edge over Random Forest in complex tabular datasets.

#### Key Performance Metrics (Dataset WITHOUT Duplicates):
* **Overall Accuracy:** **84.02%**
* **Recall (Class 1 - Cancellations):** Reached **65%** (slightly edging out Random Forest's 64%).
* **Precision (Class 1):** Solid at **74%**.
* **F1-Score (Class 1):** Maintained at **69%**.

#### Comparison to Random Forest:
XGBoost performs remarkably similarly to the Random Forest on the deduplicated dataset. While it traded a negligible fraction of overall accuracy (84.02% vs 84.15%), it managed to capture slightly more actual cancellations (65% vs 64% Recall). Both models strongly validate that non-linear approaches are essential for this problem space, and the deduplicated dataset provides a stable, realistic foundation.

## Final Evaluation & Conclusion

Throughout this modeling phase, three distinct machine learning algorithms (Logistic Regression, Random Forest, XGBoost) were evaluated across two versions of our dataset: one containing historical duplicates and one strictly deduplicated.

#### 1. The Impact of Data Quality (The Duplicate Illusion)
Across all models, the dataset containing duplicates produced artificially inflated performance metrics (e.g., Logistic Regression predicting cancellations with 65% recall vs. 47% on clean data). This confirmed the hypothesis that duplicate rows act as a form of data leakage, allowing models to "memorize" repeated bulk bookings rather than learning underlying patterns. Consequently, **the deduplicated dataset is the only reliable benchmark** for assessing real-world generalization.

#### 2. Linear vs. Non-Linear Approaches
The problem of predicting hotel cancellations is heavily dependent on complex, interacting factors. 
* The baseline **Logistic Regression** struggled on the clean dataset, managing only a **47% recall** for actual cancellations (Class 1).
* Shifting to tree-based ensemble models provided a massive leap in predictive power. Both **Random Forest** and **XGBoost** proved highly capable of mapping these non-linear relationships.

#### 3. The Top Performers: Random Forest vs. XGBoost
On the strict, deduplicated dataset, the two ensemble models delivered exceptional and closely matched results:
* **Random Forest:** 84.15% Accuracy | **64% Recall** (Class 1) | 69% F1-Score
* **XGBoost:** 84.02% Accuracy | **65% Recall** (Class 1) | 69% F1-Score

**Final Recommendation:**
While both models perform exceptionally well, **XGBoost** slightly edges out the Random Forest by capturing a marginally higher percentage of actual cancellations (65% Recall). In the business context of a hotel, accurately identifying a cancellation before it happens is highly valuable for revenue management and overbooking strategies. 
XGBoost achieved the strongest cancellation detection performance and was therefore selected as the preferred model for further optimization.
No hyperparameter tuning was performed.